# 02 - Evaluacion de Modelo: ALS Recommender

Este notebook evalua el modelo entrenado tras ejecutar `kedro run --pipeline=recommender_als`.

In [1]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

In [2]:
try:
    # Intentamos cargar la extension de Kedro para Jupyter (funciona si ejecutaste 'kedro jupyter lab')
    %load_ext kedro.ipython
except Exception:
    pass

try:
    catalog
except NameError:
    # Fallback si no se inyecto el catalogo automaticamente
    from kedro.framework.session import KedroSession
    from kedro.framework.startup import bootstrap_project

    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

    metadata = bootstrap_project(PROJECT_ROOT)
    session = KedroSession.create(project_path=PROJECT_ROOT)
    context = session.load_context()
    catalog = context.catalog
    print("Catalogo de Kedro cargado manualmente.")


[05/07/26 08:38:08] INFO     Using                                                                  ]8;id=291395;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=291396;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/framework/project/__init__.py#275\275]8;;\
                             '/mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/k                
                             edro/framework/project/rich_logging.yml' as logging configuration.                    

                    INFO     Registered line magic '%reload_kedro'                                   ]8;id=291403;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=291404;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py#64\64]8;;\

                    INFO     Registered line magic '%load_node'                                      ]8;id=291410;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=291411;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py#66\66]8;;\

                    INFO     Resolved project path as: /mnt/c/Users/ander/amazon-recsys.            ]8;id=291417;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=291418;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py#181\181]8;;\
                             To set a different path, run '%reload_kedro <project_root>'                           

[05/07/26 08:38:18] INFO     No typed parameter requirements found, returning original   ]8;id=291425;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/validation/parameter_validator.py\parameter_validator.py]8;;\:]8;id=291426;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/validation/parameter_validator.py#108\108]8;;\
                             parameters                                                                            

[05/07/26 08:38:19] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=291433;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=291434;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

[05/07/26 08:38:20] INFO     Kedro project amazon-recsys                                            ]8;id=291440;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=291441;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py#147\147]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=291447;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=291448;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py#148\148]8;;\
                             'pipelines'                                                                           

[05/07/26 08:38:22] INFO     Registered line magic 'run_viz'                                        ]8;id=291454;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=291455;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/ipython/__init__.py#154\154]8;;\

In [3]:
# Cargar metricas de evaluacion desde el catalogo
try:
    regression_metrics = catalog.load("als_regression_metrics")
    ranking_metrics = catalog.load("als_ranking_metrics")
    pop_ranking_metrics = catalog.load("popularity_ranking_metrics")
    metrics_comparison = catalog.load("als_metrics_comparison")
    print("Metricas cargadas correctamente desde el catalogo de Kedro.")
except Exception as e:
    print(f"Error al cargar del catalogo: {e}")

[05/07/26 08:39:05] INFO     Loading data from als_regression_metrics (JSONDataset)...         ]8;id=291462;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=291463;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py#1053\1053]8;;\

                    INFO     Loading data from als_ranking_metrics (JSONDataset)...            ]8;id=291468;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=291469;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py#1053\1053]8;;\

                    INFO     Loading data from popularity_ranking_metrics (JSONDataset)...     ]8;id=291474;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=291475;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py#1053\1053]8;;\

                    INFO     Loading data from als_metrics_comparison (JSONDataset)...         ]8;id=291480;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=291481;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py#1053\1053]8;;\

Metricas cargadas correctamente desde el catalogo de Kedro.


## Comparacion de Metricas

In [4]:
print("Metricas de Regresion (ALS):")
display(pd.DataFrame([regression_metrics]))

print("\nMetricas de Ranking (ALS):")
display(pd.DataFrame([ranking_metrics]))

print("\nMetricas de Ranking (Popularidad Baseline):")
display(pd.DataFrame([pop_ranking_metrics]))

Metricas de Regresion (ALS):


,rmse
0,1.602769



Metricas de Ranking (ALS):


,recall@10,recall@20
0,0.00029,0.000543



Metricas de Ranking (Popularidad Baseline):


,recall@10,recall@20
0,0.02445,0.036727


In [5]:
print("\nComparacion de Metricas:")
display(metrics_comparison)


Comparacion de Metricas:



{
    'als_regression': {'rmse': 1.602769},
    'als_ranking': {'recall@10': 0.00029, 'recall@20': 0.000543},
    'popularity_ranking': {'recall@10': 0.02445, 'recall@20': 0.036727},
    'als_beats_popularity': False
}

## Visualizacion de Recomendaciones

Exploramos las recomendaciones generadas por ALS frente al baseline de popularidad.

In [6]:
als_recs = catalog.load("als_recommendations_top_k")
pop_recs = catalog.load("popularity_recommendations_top_k")

print("Muestra de recomendaciones generadas por ALS:")
display(als_recs.limit(10).toPandas())

[05/07/26 08:40:50] INFO     Loading data from als_recommendations_top_k (SparkDataset)...     ]8;id=291486;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=291487;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py#1053\1053]8;;\

26/05/07 08:40:54 WARN Utils: Your hostname, OMEN16Ander resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/07 08:40:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/07 08:40:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[05/07/26 08:40:58] INFO     Loading data from popularity_recommendations_top_k                ]8;id=291492;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=291493;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/io/data_catalog.py#1053\1053]8;;\
                             (SparkDataset)...                                                                     

Muestra de recomendaciones generadas por ALS:


,user_idx,item_idx,score,rank
0,0,27950,5.669094,1
1,0,14090,5.373882,2
2,0,17653,5.360067,3
3,0,29562,5.337509,4
4,0,15965,5.309278,5
5,0,23870,5.290396,6
6,0,21657,5.272759,7
7,0,23935,5.265434,8
8,0,7188,5.254589,9
9,0,17496,5.230458,10


26/05/07 08:41:07 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
print("Muestra de recomendaciones por Popularidad (Baseline):")
display(pop_recs.limit(10).toPandas())

Muestra de recomendaciones por Popularidad (Baseline):


,user_idx,item_idx,score,rank
0,0,0,1346,1
1,0,2,1048,2
2,0,3,1042,3
3,0,4,954,4
4,0,1,939,5
5,0,5,868,6
6,0,6,835,7
7,0,7,798,8
8,0,10,699,9
9,0,12,687,10
